# An SIR simulation of an infectious disease
## Intro
Here, we'll set up a relatively simple SIR-based model.
It is a little more complicated than the standard three-compartment SIR model,
just so that we can add in public health interventions
and then adjust parameters to look at the effect of interventions.
The model is intended to represent some short-lived infectious
disease (such as a respiratory virus) that leads to permanent
immunity following infection.
We start off with some standard installs and imports, which you can ignore
(including the outputs printed after the cell).

In [ ]:
%pip uninstall orbax-checkpoint flax dopamine-rl --yes
%pip install summerepi2==1.3.6

In [ ]:
import logging
logging.getLogger("jax").setLevel(logging.CRITICAL)
import pandas as pd
from plotly.graph_objects import Figure
import plotly.graph_objects as go

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

## Building the model
We'll add compartments to distinguish between sequential asymptomatic
and symptomatic stages of infection, referring to these as `_asympt` and `_sympt`.
We'll also include a vaccinated compartment, which we'll come back to below.

In [ ]:
start_time = 0.0
end_time = 70.0
model_comps = [
    "vaccinated", 
    "susceptible", 
    "infectious_asympt", 
    "infectious_sympt", 
    "recovered",
]
infect_comps = [
    "infectious_asympt", 
    "infectious_sympt",
]
sir_model = CompartmentalModel(
    times=[start_time, end_time], 
    compartments=model_comps, 
    infectious_compartments=infect_comps, 
    timestep=0.2,
)
print(
    f"The simulation runs from time {round(start_time)} to {round(end_time)}. "
    f"The model compartments are called: {', '.join(model_comps)}. "
    f"Of these compartments, {', '.join(infect_comps)} are considered equally infectious."
)

### Data
Here are some synthetic data,
that were actually created by running the model,
getting the outputs, adding noise and rounding.

In [ ]:
data = pd.Series(
    {
        20.0: 187.0,
        22.0: 302.0,
        24.0: 982.0,
        26.0: 1383.0,
        28.0: 2203.0,
        30.0: 4304.0,
        32.0: 5479.0,
        34.0: 12945.0,
        36.0: 11310.0,
        38.0: 14238.0,
        40.0: 9262.0,
        42.0: 3677.0,
        44.0: 2899.0,
        46.0: 2149.0,
        48.0: 1443.0,
        50.0: 1049.0,
    },
)

## Interventions
### Face masks
Let's implement population-wide use of face masks into our simple model.
Rather than the infection rate being determined by the contact rate
parameter alone, we'll adjust this based on the proportion of people
wearing the masks ("coverage") and the efficacy of wearing the mask
on preventing transmission in those who are wearing them.

In [ ]:
mask_cov = "face_mask_coverage"
mask_effic = "face_mask_efficacy"
inf_rate = Parameter("contact_rate") * (1.0 - Parameter(mask_effic) * Parameter(mask_cov))
src = "susceptible"
dest = "infectious_asympt"
sir_model.add_infection_frequency_flow(
    name="infection", 
    contact_rate=inf_rate, 
    source=src, 
    dest=dest,
)
print(
    f"The process of infection transitions people from the {src} compartment to the {dest} compartment. "
    f"The rate of infection is determined by the coverage '{mask_cov}' and efficacy '{mask_effic}' of vaccination. "
    "The product of these two parameters determine how much face masks reduce transmission by."
)

### Vaccination
Let's also allow that vaccination can reduce the rate at which people are infected,
but only apply this to the vaccinated population.
We included a vaccinated compartment earlier,
so we'll have to apply an infection process to this compartment too,
but adjust the rate at which people from this compartment are infected
according to the efficacy of the vaccine being used.
We'll only see an effect from this if we start some of the 
population off in the vaccinated compartment, of course.

In [ ]:
vacc_effic = "vacc_efficacy"
vacc_infection_rate = inf_rate * (1.0 - Parameter(vacc_effic))
sir_model.add_infection_frequency_flow(
    name="infection_vacc", 
    contact_rate=vacc_infection_rate, 
    source="vaccinated", 
    dest="infectious_asympt",
)
print(
    "Vaccination reduces the rate of infection for the vaccinated population "
    f"according to the efficacy of the vaccine, which is given by the parameter {vacc_effic}."
)

### Case isolation
We'll now look at incorporating the effect of case isolation for infected people with symptoms.
Because we have two infectious compartments in series,
if we want the average time infectious (in the absence of case isolation)
to be the reciprocal of the recovery rate,
we'll have to double the rate of transition between the compartments.
Once we've done that, we can add an additional rate at which people with symptoms
effectively remove themselves from the infectious population,
which we can refer to as case isolation.
Because this only applies to the compartment with symptoms (i.e. the second
half of the infectious period),
we can add this on to the rate of transition from `infectious_sympt` to `recovered`.

If we're going to work out the rate of case notifications in the model,
we'll also have to track some rate of transition between compartments
because this is an incident (rather than prevalent) quantity.
So in the model, this is a rate of movement between compartments
rather than the size of a compartment.

In [ ]:
progression_rate = Parameter("recovery_rate") * 2.0
sir_model.add_transition_flow(
    name="progression", 
    fractional_rate=progression_rate, 
    source="infectious_asympt", 
    dest="infectious_sympt",
)
resolve_sympt_rate = progression_rate + Parameter("isolation_rate")
src = "infectious_sympt"
dest = "recovered"
sir_model.add_transition_flow(
    name="recovery", 
    fractional_rate=resolve_sympt_rate, 
    source=src,
    dest=dest,
)
sir_model.request_output_for_flow(
    "onset",
    "infection",
)
print(
    "For case isolation, we add an additional flow that increases "
    f"the rate at which people move from the {src} compartment "
    f"to the {dest} compartment. "
)

### Preparing the model
We'll start with a total population of one ninth of 7 million,
based on Victoria's approximate total population of 7 million 
divided by nine LPHUs.

In [ ]:
total_population = 7e6 / 9.0
infectious_seed = 1.0
suscept_pop = total_population - infectious_seed
start_pop = {
    "susceptible": suscept_pop * (1.0 - Parameter("vacc_coverage")),
    "vaccinated": suscept_pop * Parameter("vacc_coverage"),
    "infectious_asympt": infectious_seed,
}
sir_model.set_initial_population(start_pop)
print(
    f"The total population simulated to be {total_population} persons. "
    f"This population is seeded with {infectious_seed} infectious persons to trigger the epidemic. "
)

### Running the model with interventions
Now we have a model that can run three different interventions,
and any combination of those three interventions together.
This cell is for you to experiment with different values 
for the five different intervention-related input parameters.
See how they affect the epidemic and whether the results are
consistent with what you expected.

Note that if you want to change any of the structures of the model above,
you will probably need to re-run all the preceding cells,
but if you're happy with the structure, you should be able 
to adjust the parameters and re-run in this cell alone.
If things seem to have got tangled,
re-run the notebook from the start to make sure the code has all been run in the expected order.

In [ ]:
infection_parameters = {
    "contact_rate": 0.5,
    "recovery_rate": 0.2,
}
intervention_parameters = {
    "face_mask_coverage": 0.0,
    "face_mask_efficacy": 0.0,
    "isolation_rate": 0.0,
    "vacc_efficacy": 0.0,
    "vacc_coverage": 0.0,
}
case_detection_prop = 0.2
sir_model.run(infection_parameters | intervention_parameters)
cases = sir_model.get_derived_outputs_df()["onset"] * case_detection_prop

In [ ]:
fig = Figure()
fig.add_trace(go.Scatter(x=data.index, y=data, mode="markers", name="data"))
fig.add_trace(go.Scatter(x=cases.index, y=cases, name="model output"))